# Gate 0 funding pre-registration — FROZEN BEFORE DATA

**Freeze timestamp:** 2026-08-09T15:22:54Z. **Scope:** Gate 0 only. This notebook is research evidence, not production code. No repository, CI, credential, private endpoint, order path, or mainnet authority is created.

## Frozen hypothesis and formulas

- Hedge venue: Bybit linear `BTCUSDT` and `ETHUSDT`. Topology under test: **T0A only**, short Hyperliquid perpetual and long Bybit perpetual at equal, constant single-leg USD notional.
- Venue convention must be verified independently from official sources: positive funding means longs pay shorts. With that convention, the frozen funding cashflow is `d_tau = r_HL,tau - r_BY,tau`, and `F(t,H) = sum_(tau in (t,t+H]) d_tau`.
- Fee-only screened return is `R_fee(t,H) = F(t,H) - 0.0006` (6 bp once per four-leg maker round trip, never per hour). If a later verified real round-trip cost exceeds 6 bp, it may only downgrade a GO; it may never upgrade a NO-GO. Break-even cost `mean(F)` is descriptive and cannot change this decision.
- Full T0A return, not estimated here: `R_A = F + (b_A(t+H)-b_A(t)) - cost - slippage`, where `b_A = P_BY_perp - P_HL_perp` in normalized return units.
- Registered but **not tested** T0B formula: long Bybit spot and short HL perp gives `F_B = sum r_HL` and `R_B = F_B + (b_B(t)-b_B(t+H)) - cost - slippage`, where `b_B = P_HL_perp - P_BY_spot`. A T0A NO-GO cannot be changed by flipping direction or switching to T0B on this sample. Such observations are new hypotheses requiring a future independent sample.

## Frozen sample and settlement clock

- Evaluation interval: `(2025-08-09T14:00:00Z, 2026-08-09T14:00:00Z]`, exactly `START_MS=1754748000000`, `END_MS=1786284000000`. These values were fixed before the first market-data request.
- Holding horizons: `H = {1, 4, 12, 24, 72, 168}` hours. Entry time `t` is immediately after the timestamped settlement; the half-open rule `(t,t+H]` excludes a settlement at entry and includes one at exit. Only complete windows are used.
- Hyperliquid `fundingHistory` rows are hourly settled rates. They are never multiplied or divided by 8. Bybit historical rows are whole per-settlement-cycle rates: the complete value is placed at its actual settlement timestamp; legitimate non-settlement hours are zero, while a missing expected settlement is never zero. Bybit interval changes are inferred from timestamps and checked against `fundingInterval`.
- The unconditional main table uses every eligible UTC-hour entry. Every horizon is also stratified by whether its window includes a Bybit settlement, to expose settlement-phase effects; no phase may be selected after seeing results.

## Frozen metrics and decision rule

- BTC and ETH are separate series. For each symbol and H report rolling-window mean, median, 5th/95th percentiles, `P(F<0)`, fee-adjusted mean, `P(F-6bp<0)`, and break-even round-trip cost, in decimal, percent, and bp. Also report non-overlapping-window mean and count.
- The confirmatory family is fixed at 12 tests (`2 symbols x 6 horizons`). Family-wise alpha is 0.05 using Bonferroni two-sided 99.583333% intervals. Horizons with `N_eff=floor(complete_hours/H)<30` remain in the denominator and are labelled insufficient evidence.
- Uncertainty uses moving-block bootstrap of the underlying hourly cashflow series with 336-hour blocks, 10,000 resamples, and RNG seed 20260809. A symbol/H is significantly positive only if the fee-adjusted interval lower bound is above zero; significantly negative only if the upper bound is below zero; otherwise it is near zero.
- Overall classification: any corrected significantly positive registered test => `significantly positive — audit for omitted legs/costs`; all 12 corrected intervals below zero => `significantly negative`; otherwise => `near zero / insufficient evidence`. This is a fee-only screen, not proof of full Gate 0 GO.

## Evidence, integrity, and fail-closed rules

- Official public endpoints only; no API key: HL `POST https://api.hyperliquid.xyz/info` with `type=fundingHistory`; Bybit `GET https://api.bybit.com/v5/market/funding/history` with `category=linear`; instrument interval from `/v5/market/instruments-info`.
- Official sources checked 2026-08-10 Melbourne / 2026-08-09 UTC: [HL funding mechanics](https://hyperliquid.gitbook.io/hyperliquid-docs/trading/funding), [HL fundingHistory](https://hyperliquid.gitbook.io/hyperliquid-docs/for-developers/api/info-endpoint/perpetuals), [HL rate limits](https://hyperliquid.gitbook.io/hyperliquid-docs/for-developers/api/rate-limits-and-user-limits), [HL fees](https://hyperliquid.gitbook.io/hyperliquid-docs/trading/fees), [Bybit funding history](https://bybit-exchange.github.io/docs/v5/market/history-fund-rate), [Bybit funding sign and settlement](https://www.bybit.com/en/help-center/article/Funding-fee-calculation), [Bybit funding calculation](https://www.bybit.com/en/help-center/article/Introduction-to-Funding-Rate), [Bybit instrument interval](https://bybit-exchange.github.io/docs/v5/market/instrument), [Bybit API limits](https://bybit-exchange.github.io/docs/v5/rate-limit), and [Bybit fees](https://www.bybit.com/en/help-center/article/Bybit-Fees-You-Need-to-Know?category=b9ff53f160978e761c).
- Independent sign evidence: HL and Bybit each state that a positive funding rate makes longs pay shorts. HL states that its 8-hour formula is paid hourly at one eighth and the payment occurs at interval end; `fundingHistory.fundingRate` is therefore used once per returned hour without scaling. Bybit states that the historical rate timestamp is when the cycle settled and that the cycle rate is exchanged at that timestamp; each returned value is used once, not annualized. HL `fundingHistory.time` being interval-end remains fail-closed until the registered live boundary observation below resolves it.
- Before analysis, assert numeric fields, timestamps, unique key `(venue,symbol,funding_timestamp)`, expected HL hourly cadence, observed Bybit cadence, exact sample bounds, and no unexplained gaps. Rate magnitude and `P(r_HL>0)>0.5` are warnings that force manual sign review, not truth assumptions. Any unexplained gap, schema change, sign ambiguity, timestamp ambiguity, or repeated rate-limit failure stops immediately and is reported.
- One schema-only pull of at most 48 hours is allowed after this freeze to verify fields, ordering, boundary behavior, units, and timestamps. It must not compute cross-venue differences or statistics. The main raw evidence is append-only `gate0_funding_raw.jsonl.gz`; retries deduplicate and merge, never replace. Analysis reads the frozen file only.
- Raw manifest fields recorded here after collection: exact endpoints/parameters, retrieval time, response/page counts, raw/deduplicated row counts, first/last timestamps, and final SHA-256. No separate manifest file.
- Schema probe manifest: four market probes (HL/Bybit x BTC/ETH) each used `1786111200000..1786284000000`, exactly 48 hours; two additional instrument responses; interim gzip SHA-256 `2256f149820f4e7933827b64ead58b89f59ad84de1e5a1429c3cf58e048b57be`. HL returned 48 ascending rows per coin with fields `coin,fundingRate,premium,time`; Bybit returned 6 descending settled rows per coin with fields `fundingRate,fundingRateTimestamp,symbol`; instrument metadata returned `fundingInterval=480`, `contractType=LinearPerpetual`, `settleCoin=USDT`, `status=Trading`. No cross-venue statistic was computed.
- Final raw manifest: retrieval `2026-08-09T15:27:09.524904Z..2026-08-09T15:32:51.803715Z`; 124 unique request records = 4 schema probes + 2 instrument metadata + 118 main pages. Main pages: HL BTC 53, HL ETH 53, Bybit BTCUSDT 6, Bybit ETHUSDT 6. After boundary normalization and deduplication: HL BTC 8,760 and ETH 8,760 contiguous hourly rows from `1754751600000` through `1786284000000`; Bybit BTCUSDT 1,095 and ETHUSDT 1,095 rows from `1754755200000` through `1786262400000`, exclusively 8-hour gaps. Gzip size 298,853 bytes; final SHA-256 `4896c59d7884b74083064214581f8169c2af1093e2431fd3c88dd15f4386c4b5`.
- Implementation boundary decision: normalize HL timestamps down to their UTC hour after asserting offset `<10s`; observed main-sample `max_jitter_ms=261`. Request each chunk with 10s of right-boundary overlap, then deduplicate. Analysis still applies the frozen normalized interval `(START_MS,END_MS]`. Bybit exact-hour timestamps require no normalization.
- Single-venue sign/unit warnings, computed before any cross-venue statistic: `P(r_HL>0)` is BTC `7154/8760=81.6667%`, ETH `7272/8760=83.0137%`, so the registered >50% warning passes. Longest identical-rate runs are BTC 269h and ETH 270h; both equal `1.25e-5` per hour, the documented hourly interest component. These are records, not hard assertions.
- Public no-tier maker schedules are HL 1.5 bp/fill and Bybit 2 bp/fill, implying 7 bp for four fills. The user-specified 6 bp remains the primary screen; 7 bp is a one-way downgrade check only. Raw evidence is SEALED at the recorded SHA; any append requires an explicit re-freeze stating whether statistics were already seen.

## Not estimated in this notebook

Cross-venue basis, entry/exit slippage, USDC/USDT price divergence, margin capital cost, transfers, unequal or changing notional conventions, per-capital annualization, and account-level funding-ledger reconciliation. Public settled rates are not proof of a particular account's realized receipt. These items are not zero; they are outside this funding-only request.

## Gate 0 Codex↔Claude workflow

Milestones are pre-registration, official-source/sign verification, raw-data integrity, statistical result, and final conclusion. Each review packet states verified facts, proposed decision, uncertainty, and a precise review question. One review plus one evidence response is allowed; a blocker closes only when its author confirms or the user decides. Fail-closed events are reported immediately. Any change to a frozen item is a timestamped re-freeze recording whether results were already observed; post-result changes are a second hypothesis test. This protocol ends with Gate 0.

**Re-freeze log:** none. **Raw evidence SHA-256:** `4896c59d7884b74083064214581f8169c2af1093e2431fd3c88dd15f4386c4b5` (analysis is permitted only while the file matches this value).

**Environment decision log — 2026-08-09T15:27Z:** The first schema request returned zero market data and failed during TLS verification because Python 3.14's configured CA path did not exist. The retry uses an explicit verified context with Apple system bundle `/etc/ssl/cert.pem`, SHA-256 `9dae8d76e55cb08991f2b672d58999ea15560d910759c16b544f843bdffbb994`. Certificate verification and hostname checking remain enabled; unverified contexts, `verify=False`, `curl -k`, `PYTHONHTTPSVERIFY=0`, and a global `SSL_CERT_FILE` override are forbidden. This is an environment repair, not a re-freeze, and the registered probe parameters are unchanged.

**HL timestamp test T — PASS, 2026-08-09T16:00:10.065169Z:** A separate, non-sample BTC `fundingHistory` request made 10.065 seconds after the UTC boundary returned latest `time=1786291200032` (`16:00:00.032Z`), only 32ms after the just-passed hour. Under the pre-approved rule this proves that `time` is the interval-end settlement event, not the next interval's start. The semantic milestone closed before any cross-venue statistic was computed.

**Statistical implementation decision log — 2026-08-10 Melbourne:** Before final classification, an independent convolution audit found that binary floating-point sums misclassified theoretically zero windows as slightly positive or negative. Rates and window sums were changed to exact integer units of `1e-10`; only `P(F<0)` changed, while means, quantiles, bootstrap intervals, and classifications did not. Two complete runs were byte-identical with result JSON SHA-256 `0e93b77ad98b70694247078417968dd15e9be538439feb21acb220f777ff45e8`. This mechanical precision correction is not a re-freeze.


In [ ]:
from pathlib import Path

START_MS = 1754748000000
END_MS = 1786284000000
HORIZONS = (1, 4, 12, 24, 72, 168)
SYMBOLS = ("BTC", "ETH")
ROUND_TRIP_COST = 0.0006
PUBLIC_SCHEDULE_COST = 0.0007
RATE_SCALE = 10_000_000_000
ROUND_TRIP_COST_UNITS = 6_000_000
PUBLIC_SCHEDULE_COST_UNITS = 7_000_000
FAMILY_SIZE = 12
CI_LEVEL = 1 - 0.05 / FAMILY_SIZE
BOOTSTRAP_BLOCK_HOURS = 336
BOOTSTRAP_RESAMPLES = 10_000
RNG_SEED = 20260809
RAW_PATH = Path("gate0_funding_raw.jsonl.gz")


In [ ]:
import gzip
import hashlib
import json
import os
import ssl
import time
from datetime import datetime, timezone
from decimal import Decimal
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen

HL_INFO_URL = "https://api.hyperliquid.xyz/info"
BYBIT_BASE_URL = "https://api.bybit.com"
USER_AGENT = "gate0-funding-research/1.0"
CA_BUNDLE = Path("/etc/ssl/cert.pem")
assert CA_BUNDLE.exists()
assert hashlib.sha256(CA_BUNDLE.read_bytes()).hexdigest() == "9dae8d76e55cb08991f2b672d58999ea15560d910759c16b544f843bdffbb994"
assert os.environ.get("PYTHONHTTPSVERIFY") != "0"
assert "SSL_CERT_FILE" not in os.environ
TLS_CONTEXT = ssl.create_default_context(cafile=str(CA_BUNDLE))
assert TLS_CONTEXT.verify_mode == ssl.CERT_REQUIRED
assert TLS_CONTEXT.check_hostname is True

def canonical_json(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"), ensure_ascii=False)

def request_id(spec):
    return hashlib.sha256(canonical_json(spec).encode()).hexdigest()

def read_raw_records():
    if not RAW_PATH.exists():
        return []
    with gzip.open(RAW_PATH, "rt", encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]

def append_raw(record):
    with gzip.open(RAW_PATH, "at", encoding="utf-8") as handle:
        handle.write(canonical_json(record) + "\n")

def fetch_json(spec, attempts=3):
    rid = request_id(spec)
    cached = {row["request_id"]: row for row in read_raw_records()}
    if rid in cached:
        return cached[rid]["response"]
    last_error = None
    for attempt in range(attempts):
        try:
            if spec["method"] == "POST":
                body = canonical_json(spec["payload"]).encode()
                request = Request(spec["url"], data=body, method="POST", headers={"Content-Type": "application/json", "User-Agent": USER_AGENT})
            else:
                url = spec["url"] + "?" + urlencode(spec["params"])
                request = Request(url, method="GET", headers={"User-Agent": USER_AGENT})
            with urlopen(request, timeout=30, context=TLS_CONTEXT) as response:
                payload = json.loads(response.read())
                status = response.status
            record = {"record_kind": spec["record_kind"], "request_id": rid, "venue": spec["venue"], "symbol": spec["symbol"], "request": spec, "retrieved_at_utc": datetime.now(timezone.utc).isoformat(), "http_status": status, "response": payload}
            append_raw(record)
            return payload
        except (HTTPError, URLError, TimeoutError, json.JSONDecodeError) as exc:
            last_error = exc
            if attempt + 1 < attempts:
                time.sleep(2 ** attempt)
    raise RuntimeError(f"fail-closed public API request failed: {spec['venue']} {spec['symbol']}") from last_error


In [ ]:
# Schema-only probe: <=48 hours, no cross-venue difference or performance statistic.
PROBE_START_MS = END_MS - 48 * 3_600_000
schema_probe = {}
for coin in SYMBOLS:
    bybit_symbol = coin + "USDT"
    hl_spec = {"record_kind": "schema_probe", "venue": "hyperliquid", "symbol": coin, "method": "POST", "url": HL_INFO_URL, "payload": {"type": "fundingHistory", "coin": coin, "startTime": PROBE_START_MS, "endTime": END_MS}}
    bybit_spec = {"record_kind": "schema_probe", "venue": "bybit", "symbol": bybit_symbol, "method": "GET", "url": BYBIT_BASE_URL + "/v5/market/funding/history", "params": {"category": "linear", "symbol": bybit_symbol, "startTime": PROBE_START_MS, "endTime": END_MS, "limit": 200}}
    instrument_spec = {"record_kind": "instrument_metadata", "venue": "bybit", "symbol": bybit_symbol, "method": "GET", "url": BYBIT_BASE_URL + "/v5/market/instruments-info", "params": {"category": "linear", "symbol": bybit_symbol}}
    schema_probe[("hyperliquid", coin)] = fetch_json(hl_spec)
    schema_probe[("bybit", bybit_symbol)] = fetch_json(bybit_spec)
    schema_probe[("bybit_instrument", bybit_symbol)] = fetch_json(instrument_spec)

for (venue, symbol), payload in schema_probe.items():
    if venue == "hyperliquid":
        rows = payload
        times = [int(row["time"]) for row in rows]
    elif venue == "bybit":
        assert payload["retCode"] == 0, payload
        rows = payload["result"]["list"]
        times = [int(row["fundingRateTimestamp"]) for row in rows]
    else:
        assert payload["retCode"] == 0, payload
        rows = payload["result"]["list"]
        times = []
    order = "ascending" if times == sorted(times) else "descending" if times == sorted(times, reverse=True) else "unordered"
    print({"venue": venue, "symbol": symbol, "rows": len(rows), "fields": sorted(rows[0]) if rows else [], "first_ts": times[0] if times else None, "last_ts": times[-1] if times else None, "order": order})


In [ ]:
# RED integrity test: this must fail before the frozen raw evidence exists.
assert RAW_PATH.exists(), "RED: frozen raw evidence has not been created"
assert RAW_PATH.stat().st_size > 0, "RED: frozen raw evidence is empty"
probe_records = [row for row in read_raw_records() if row["record_kind"] in {"schema_probe", "instrument_metadata"}]
assert len(probe_records) == 6, f"expected 6 unique schema records, got {len(probe_records)}"
assert len({row["request_id"] for row in probe_records}) == 6
for row in probe_records:
    assert row["http_status"] == 200
    if row["venue"] == "hyperliquid":
        values = row["response"]
        assert values and set(values[0]) == {"coin", "fundingRate", "premium", "time"}
        assert all(abs(int(v["time"]) % 3_600_000) < 10_000 for v in values)
        assert all(float(v["fundingRate"]) == float(v["fundingRate"]) for v in values)
    elif row["record_kind"] == "schema_probe":
        values = row["response"]["result"]["list"]
        assert values and {"symbol", "fundingRate", "fundingRateTimestamp"} <= set(values[0])
        assert all(int(v["fundingRateTimestamp"]) % 3_600_000 == 0 for v in values)
        assert all(float(v["fundingRate"]) == float(v["fundingRate"]) for v in values)
    else:
        values = row["response"]["result"]["list"]
        assert len(values) == 1 and int(values[0]["fundingInterval"]) > 0
print("Schema evidence integrity: PASS")


In [ ]:
# Append-only collection of the registered 12-month sample. No statistics are computed here.
HL_CHUNK_MS = 7 * 24 * 3_600_000
for coin in SYMBOLS:
    cursor = START_MS
    page = 0
    while cursor < END_MS:
        chunk_end = min(cursor + HL_CHUNK_MS, END_MS)
        spec = {"record_kind": "main_page", "venue": "hyperliquid", "symbol": coin, "method": "POST", "url": HL_INFO_URL, "payload": {"type": "fundingHistory", "coin": coin, "startTime": cursor, "endTime": chunk_end + 10_000}}
        rows = fetch_json(spec)
        assert rows, f"fail-closed empty HL page: {coin} {cursor}..{chunk_end}"
        print("HL page", coin, page, cursor, chunk_end, len(rows))
        cursor = chunk_end
        page += 1
        time.sleep(1.6)

for coin in SYMBOLS:
    symbol = coin + "USDT"
    cursor = END_MS
    for page in range(20):
        spec = {"record_kind": "main_page", "venue": "bybit", "symbol": symbol, "method": "GET", "url": BYBIT_BASE_URL + "/v5/market/funding/history", "params": {"category": "linear", "symbol": symbol, "endTime": cursor, "limit": 200}}
        payload = fetch_json(spec)
        assert payload["retCode"] == 0, payload
        rows = payload["result"]["list"]
        assert rows, f"fail-closed empty Bybit page before sample start: {symbol} {cursor}"
        times = [int(row["fundingRateTimestamp"]) for row in rows]
        assert times == sorted(times, reverse=True), f"fail-closed Bybit order changed: {symbol}"
        print("Bybit page", symbol, page, max(times), min(times), len(rows))
        if min(times) <= START_MS:
            break
        next_cursor = min(times) - 1
        assert next_cursor < cursor
        cursor = next_cursor
        time.sleep(0.25)
    else:
        raise AssertionError(f"fail-closed Bybit pagination did not reach sample start: {symbol}")


In [ ]:
# RED full-coverage test: fails until every registered main-data page is collected.
main_records = [row for row in read_raw_records() if row["record_kind"] == "main_page"]
assert main_records, "RED: no registered 12-month main-data pages have been collected"
hour_ms = 3_600_000
expected_hours = list(range(START_MS + hour_ms, END_MS + hour_ms, hour_ms))
coverage = {}
for coin in SYMBOLS:
    keyed = {}
    for record in main_records:
        if record["venue"] != "hyperliquid" or record["symbol"] != coin:
            continue
        for value in record["response"]:
            raw_time = int(value["time"])
            offset = raw_time % hour_ms
            assert offset < 10_000, f"fail-closed HL timestamp offset: {coin} {raw_time}"
            normalized = raw_time - offset
            if START_MS < normalized <= END_MS:
                rate = Decimal(value["fundingRate"])
                assert rate * RATE_SCALE == int(rate * RATE_SCALE)
                if normalized in keyed:
                    assert keyed[normalized] == rate, f"conflicting duplicate HL rate: {coin} {normalized}"
                keyed[normalized] = rate
    assert sorted(keyed) == expected_hours, f"fail-closed unexplained HL gap: {coin} got {len(keyed)} expected {len(expected_hours)}"
    coverage[("hyperliquid", coin)] = keyed

for coin in SYMBOLS:
    symbol = coin + "USDT"
    keyed = {}
    for record in main_records:
        if record["venue"] != "bybit" or record["symbol"] != symbol:
            continue
        for value in record["response"]["result"]["list"]:
            ts = int(value["fundingRateTimestamp"])
            assert ts % hour_ms == 0, f"fail-closed Bybit non-hour timestamp: {symbol} {ts}"
            if START_MS < ts <= END_MS:
                rate = Decimal(value["fundingRate"])
                assert rate * RATE_SCALE == int(rate * RATE_SCALE)
                if ts in keyed:
                    assert keyed[ts] == rate, f"conflicting duplicate Bybit rate: {symbol} {ts}"
                keyed[ts] = rate
    times = sorted(keyed)
    assert times, f"fail-closed no Bybit sample: {symbol}"
    gaps = [(b - a) // hour_ms for a, b in zip(times, times[1:])]
    assert all((b - a) % hour_ms == 0 and 0 < (b - a) <= 8 * hour_ms for a, b in zip(times, times[1:])), f"fail-closed unexplained Bybit gap: {symbol}"
    assert times[0] - START_MS <= 8 * hour_ms and END_MS - times[-1] < 8 * hour_ms
    coverage[("bybit", coin)] = keyed
    print({"venue": "bybit", "symbol": symbol, "dedup_rows": len(keyed), "first_ts": times[0], "last_ts": times[-1], "gap_hours": sorted(set(gaps))})

for coin in SYMBOLS:
    times = sorted(coverage[("hyperliquid", coin)])
    print({"venue": "hyperliquid", "symbol": coin, "dedup_rows": len(times), "first_ts": times[0], "last_ts": times[-1], "gap_hours": [1]})
print("Frozen 12-month raw coverage integrity: PASS")


In [ ]:
# Single-venue diagnostics only; deliberately before any cross-venue calculation.
FROZEN_RAW_SHA = "4896c59d7884b74083064214581f8169c2af1093e2431fd3c88dd15f4386c4b5"
assert hashlib.sha256(RAW_PATH.read_bytes()).hexdigest() == FROZEN_RAW_SHA
assert RAW_PATH.stat().st_mode & 0o222 == 0, "fail-closed raw evidence is not SEALED"
max_jitter_ms = max(int(value["time"]) % hour_ms for record in main_records if record["venue"] == "hyperliquid" for value in record["response"])
assert max_jitter_ms == 261
for record in probe_records:
    if record["record_kind"] == "schema_probe":
        request = record["request"].get("payload", record["request"].get("params"))
        assert request["endTime"] - request["startTime"] == 48 * hour_ms
for coin in SYMBOLS:
    values = [coverage[("hyperliquid", coin)][ts] for ts in expected_hours]
    p_positive = sum(value > 0 for value in values) / len(values)
    assert p_positive > 0.5, f"fail-closed HL sign warning: {coin}"
    longest = current = 0
    longest_value = previous = None
    for value in values:
        current = current + 1 if value == previous else 1
        previous = value
        if current > longest:
            longest, longest_value = current, value
    print({"symbol": coin, "p_hl_positive": p_positive, "longest_identical_hours": longest, "longest_value": longest_value})
print("Single-venue sign and unit warnings: PASS")


In [ ]:
# RED semantic barrier: test T must set this to period_end before any F is computed.
HL_TIMESTAMP_SEMANTICS = "period_end"
assert HL_TIMESTAMP_SEMANTICS == "period_end", "RED: HL fundingHistory.time semantics unresolved; cross-venue statistics forbidden"


In [ ]:
import numpy as np

def rolling_sum(values, hours):
    prefix = np.concatenate((np.zeros(1, dtype=values.dtype), np.cumsum(values)))
    return prefix[hours:] - prefix[:-hours]

def describe(values):
    return {"mean": float(np.mean(values)), "median": float(np.median(values)), "q05": float(np.quantile(values, .05)), "q95": float(np.quantile(values, .95)), "p_negative": float(np.mean(values < 0))}

def bootstrap_means(values, rng, batch_size=100):
    n, block = len(values), BOOTSTRAP_BLOCK_HOURS
    blocks = (n + block - 1) // block
    result = np.empty((BOOTSTRAP_RESAMPLES, len(HORIZONS)))
    offsets = np.arange(block)
    for first in range(0, BOOTSTRAP_RESAMPLES, batch_size):
        size = min(batch_size, BOOTSTRAP_RESAMPLES - first)
        starts = rng.integers(0, n - block + 1, size=(size, blocks))
        sample = values[(starts[:, :, None] + offsets).reshape(size, -1)][:, :n]
        prefix = np.concatenate((np.zeros((size, 1)), np.cumsum(sample, axis=1)), axis=1)
        for column, hours in enumerate(HORIZONS):
            result[first:first + size, column] = np.mean(prefix[:, hours:] - prefix[:, :-hours], axis=1)
    return result

assert np.array_equal(rolling_sum(np.array([1, 2, 3], dtype=np.int64), 2), np.array([3, 5]))
assert rolling_sum(np.array([1, -1], dtype=np.int64), 2)[0] == 0
assert int(Decimal("0.0006") * RATE_SCALE) == ROUND_TRIP_COST_UNITS
rng = np.random.default_rng(RNG_SEED)
results, phase_results = [], []
tail = (1 - CI_LEVEL) / 2
for coin in SYMBOLS:
    hl_units = np.array([int(coverage[("hyperliquid", coin)][ts] * RATE_SCALE) for ts in expected_hours], dtype=np.int64)
    by_units = np.array([int(coverage[("bybit", coin)].get(ts, Decimal(0)) * RATE_SCALE) for ts in expected_hours], dtype=np.int64)
    event = np.array([ts in coverage[("bybit", coin)] for ts in expected_hours])
    differential_units = hl_units - by_units
    differential = differential_units / RATE_SCALE
    bootstrap = bootstrap_means(differential, rng)
    for column, hours in enumerate(HORIZONS):
        funded_units = rolling_sum(differential_units, hours)
        funded = funded_units / RATE_SCALE
        row = {"symbol": coin, "hours": hours, "n": len(funded), "n_eff": len(differential) // hours, **describe(funded)}
        row.update({"mean_after_6bp": row["mean"] - ROUND_TRIP_COST, "p_after_6bp_negative": float(np.mean(funded_units < ROUND_TRIP_COST_UNITS)), "nonoverlap_mean": float(np.mean(funded[::hours])), "nonoverlap_n": len(funded[::hours])})
        ci = np.quantile(bootstrap[:, column], [tail, 1 - tail])
        row.update({"ci6_low": float(ci[0] - ROUND_TRIP_COST), "ci6_high": float(ci[1] - ROUND_TRIP_COST), "mean_after_7bp": row["mean"] - PUBLIC_SCHEDULE_COST, "ci7_low": float(ci[0] - PUBLIC_SCHEDULE_COST), "ci7_high": float(ci[1] - PUBLIC_SCHEDULE_COST)})
        row["class6"] = "positive" if row["ci6_low"] > 0 else "negative" if row["ci6_high"] < 0 else "near_zero"
        row["class7"] = "positive" if row["ci7_low"] > 0 else "negative" if row["ci7_high"] < 0 else "near_zero"
        results.append(row)
        contains = rolling_sum(event.astype(np.int64), hours) > 0
        for label, mask in (("contains_bybit_settlement", contains), ("no_bybit_settlement", ~contains)):
            if np.any(mask): phase_results.append({"symbol": coin, "hours": hours, "phase": label, "n": int(np.sum(mask)), **describe(funded[mask])})
def overall(rows, key):
    labels = [row[key] for row in rows]
    return "significantly_positive" if "positive" in labels else "significantly_negative" if all(label == "negative" for label in labels) else "near_zero_or_insufficient"
print(json.dumps({"results": results, "phase_results": phase_results, "overall6": overall(results, "class6"), "overall7": overall(results, "class7")}, separators=(",", ":")))


# Reviewed Gate 0 result

**Headline: zero of the 12 frozen tests is significantly positive.** H <= 72 is significantly negative for both assets; H=168 is statistically indistinguishable from zero. The horizons are six deterministic aggregations of two asset series, not 12 independent pieces of economic evidence. Values below are bp per equal, constant single-leg notional.

| Asset | H | N | Mean | Median | 5% | 95% | P(F<0) | Mean after 6bp | P(after 6bp<0) | Corrected CI after 6bp | Class |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---|
| BTC | 1 | 8760 | 0.0362 | 0.1169 | -0.3390 | 0.1250 | 24.3721% | -5.9638 | 100.0000% | [-5.9785, -5.9512] | negative |
| BTC | 4 | 8757 | 0.1448 | 0.2200 | -0.5459 | 0.6140 | 33.2306% | -5.8552 | 100.0000% | [-5.9141, -5.8047] | negative |
| BTC | 12 | 8749 | 0.4335 | 0.5000 | -1.1009 | 1.6098 | 27.6603% | -5.5665 | 100.0000% | [-5.7423, -5.4138] | negative |
| BTC | 24 | 8737 | 0.8649 | 0.9514 | -1.7455 | 2.8735 | 21.3117% | -5.1351 | 99.7940% | [-5.4855, -4.8274] | negative |
| BTC | 72 | 8689 | 2.5998 | 2.8279 | -2.9929 | 7.0399 | 17.0215% | -3.4002 | 87.2828% | [-4.4590, -2.4933] | negative |
| BTC | 168 | 8593 | 6.0306 | 6.7015 | -4.1847 | 13.4165 | 11.5094% | +0.0306 | 46.0840% | [-2.3928, +2.2116] | near zero |
| ETH | 1 | 8760 | 0.0441 | 0.1250 | -0.3708 | 0.1651 | 22.7055% | -5.9559 | 99.9886% | [-5.9756, -5.9399] | negative |
| ETH | 4 | 8757 | 0.1762 | 0.2728 | -0.5951 | 0.7593 | 31.4948% | -5.8238 | 99.9772% | [-5.9023, -5.7596] | negative |
| ETH | 12 | 8749 | 0.5277 | 0.6113 | -1.1433 | 1.8574 | 24.7343% | -5.4723 | 99.9657% | [-5.7077, -5.2785] | negative |
| ETH | 24 | 8737 | 1.0537 | 1.1970 | -1.7961 | 3.3810 | 18.7250% | -4.9463 | 99.6223% | [-5.4167, -4.5547] | negative |
| ETH | 72 | 8689 | 3.1580 | 3.5414 | -3.2359 | 8.1256 | 16.0893% | -2.8420 | 81.9312% | [-4.2501, -1.6615] | negative |
| ETH | 168 | 8593 | 7.2730 | 7.9065 | -4.4680 | 18.2944 | 12.8477% | +1.2730 | 38.7990% | [-1.8913, +4.1163] | near zero |

Point estimates after the user-specified 6bp cost are positive only at 168h: BTC `+0.0306bp` and ETH `+1.2730bp`; both corrected intervals cross zero. Under the public no-tier 7bp maker schedule, BTC168 becomes `-0.9694bp` and ETH168 is `+0.2730bp`; both remain near zero.

| Asset | 6bp break-even | 7bp break-even | Pre-cost annualized per-notional |
|---|---:|---:|---:|
| BTC | 165.532h | 193.121h | 3.1752% |
| ETH | 136.167h | 158.861h | 3.8600% |

The only positive grid point is the maximum registered H=168; BTC is only 2.468h beyond its 6bp break-even. This is a boundary result. No inference is made for H>168; a longer hold is a new hypothesis requiring a new freeze and independent sample.

Settlement-phase means (contains / does not contain a Bybit settlement) are BTC H1 `-0.2100/+0.0714bp`, H4 `+0.0048/+0.2848bp`; ETH H1 `-0.1646/+0.0739bp`, H4 `+0.0572/+0.2953bp`. Every H>=12 window contains a settlement. Non-overlapping means/counts are BTC `[0.0362/8760, 0.1450/2190, 0.4350/730, 0.8699/365, 2.6196/121, 6.0820/52]` and ETH `[0.0441/8760, 0.1763/2190, 0.5288/730, 1.0575/365, 3.1690/121, 7.3908/52]`.

Per-capital annualization is not identified: `APR_capital = APR_single_leg_notional / (m_HL + m_BY + buffers)`. No margin fractions or leverage were frozen. A smaller denominator does not improve the underlying return and worsens liquidation distance, margin-call frequency, and buffer needs.

**Gate 0 economic classification: near zero — continue only as pure execution-layer learning, not as a tradeable positive-expectation GO.** Basis, exit slippage, stablecoin divergence, capital cost, and transfers are omitted negative costs, so full-return expectation is more likely worse. Account eligibility remains a separate hard prerequisite that only the account holder can verify from both venues' current terms.
